In [1]:
from dotenv import load_dotenv
from openai import OpenAI
import json
import os

In [2]:
load_dotenv(override=True)

True

In [ ]:

groq_api_key = os.getenv('GROQ_API_KEY')
openai = OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1"
)


In [4]:
model = "openai/gpt-oss-120b"

In [21]:
mood_history = []
recommendations = []

In [6]:
def log_mood(mood: str) -> str:
    mood_history.append(mood)
    return f" Mood logged: {mood}"

In [7]:
def save_song(mood: str, song: str, artist: str) -> str:  
    rec = {
        "mood": mood,
        "song": song,
        "artist": artist
    }
    recommendations.append(rec)
    return f" Saved: {song} by {artist} for {mood} mood"

In [8]:
def get_mood_stats(mood: str) -> str:
    count = mood_history.count(mood)
    return f"You've been {mood} {count} times"

In [9]:
def get_last_song() -> str:
    if recommendations:
        last = recommendations[-1]
        return f"Last time you were {last['mood']}, I recommended {last['song']} by {last['artist']}"
    return "No recommendations yet"

In [10]:
log_mood_json = {
    "name": "log_mood",
    "description": "Log the user's current mood",
    "parameters": {
        "type": "object",
        "properties": {
            "mood": {"type": "string", "description": "The mood (happy, sad, angry, relaxed, etc)"}
        },
        "required": ["mood"]
    }
}


In [11]:
save_song_json = {
    "name": "save_song",
    "description": "Save a song recommendation",
    "parameters": {
        "type": "object",
        "properties": {
            "mood": {"type": "string"},
            "song": {"type": "string"},
            "artist": {"type": "string"}
        },
        "required": ["mood", "song", "artist"]
    }
}


In [12]:
get_mood_stats_json = {
    "name": "get_mood_stats",
    "description": "Check how many times a mood occurred",
    "parameters": {
        "type": "object",
        "properties": {
            "mood": {"type": "string"}
        },
        "required": ["mood"]
    }
}

In [13]:
get_last_song_json = {
    "name": "get_last_song",
    "description": "Get the last recommended song",
    "parameters": {"type": "object", "properties": {}}
}

In [14]:
tools = [
    {"type": "function", "function": log_mood_json},
    {"type": "function", "function": save_song_json},
    {"type": "function", "function": get_mood_stats_json},
    {"type": "function", "function": get_last_song_json}
]

In [16]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f" {tool_name}")
        
        tool_func = globals().get(tool_name)
        if tool_func:
            result = tool_func(**arguments)
            results.append({
                "role": "tool",
                "content": json.dumps({"result": result}),
                "tool_call_id": tool_call.id
            })
    return results

In [17]:
def mood_agent(user_input):
    system = "You recommend songs based on mood. Log the mood, suggest 2 songs, save them, and mention if you've seen this mood before."
    
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user_input}
    ]
    
    done = False
    while not done:
        response = openai.chat.completions.create(
            model=model,
            messages=messages,
            tools=tools
        )
        
        if response.choices[0].finish_reason == "tool_calls":
            messages.append(response.choices[0].message)
            results = handle_tool_calls(response.choices[0].message.tool_calls)
            messages.extend(results)
        else:
            done = True
    
    return response.choices[0].message.content

In [22]:
print(mood_agent("I'm feeling happy today, recommend some songs"))
print("\n" + mood_agent("Now I'm sad, need music"))

 get_mood_stats
 log_mood
 save_song
 save_song
I see you’re feeling **happy** today! 🎉 This is the first time I’ve logged this mood for you.

Here are two upbeat tracks to match your vibe:

1. **“Happy” – Pharrell Williams**  
2. **“Walking on Sunshine” – Katrina & The Waves**

Enjoy the sunshine! Let me know if you’d like more recommendations or a different mood playlist.
 log_mood
 get_mood_stats
 save_song
 save_song

I’m sorry you’re feeling sad. I’ve logged that mood for you, and it looks like you’ve been in a sad mood before (the system shows it’s happened once already). Here are two songs that might lift your spirits:

1. **“Someone Like You” – Adele**  
2. **“Fix You” – Coldplay**  

I’ve saved both recommendations under your “sad” mood. I hope they bring you a little comfort. 🎧💙
